# Imports

In [ ]:
!pip install pingouin
!pip install qlatent
%pip install --quiet git+https://github.com/cnai-lab/qpsychometric.git


In [ ]:
import torch
import pandas as pd
from pathlib import Path
import gc
from tqdm.auto import tqdm
import warnings
import pingouin as pg
from sentence_transformers import SentenceTransformer, util
from qlatent.qmnli.qmnli import *
from qlatent.qmnli.qmnli import _QMNLI, QMNLI
device = 0 if torch.cuda.is_available() else -1
print(device)

In [ ]:
softmax_files = [True, False]

def split_question(Q, index, scales, softmax, filters):
  result = []
  for s in scales:
    q = QCACHE(Q())
    for sf in softmax:
      for f in filters:
        if sf:
            qsf = QSOFTMAX(q,dim=[index[0], s])
            qsf_f = QFILTER(qsf,filters[f],filtername=f)
            print((index, s),sf,f)
            result.append(qsf_f)

            qsf = QSOFTMAX(q,dim=s)
            qsf_f = QFILTER(qsf,filters[f],filtername=f)
            print(s,sf,f)
            result.append(qsf_f)

            qsf = QSOFTMAX(q,dim=index[0])
            qsf_f = QFILTER(qsf,filters[f],filtername=f)
            print(index[0],sf,f)
            result.append(qsf_f)
        else:
            qsf = QPASS(q,descupdate={'softmax':''})
            qsf_f = QFILTER(qsf,filters[f],filtername=f)
            print(s,sf,f)
            result.append(qsf_f)
  return result


def print_permutations(q):
#     for q in Q1s:
    W = q._pdf['W']
    print(q._descriptor)
    for i, (kmap, w) in enumerate(zip(q._keywords_map, W)):
        context = q._context_template.format_map(kmap)
        answer = q._answer_template.format_map(kmap)
        print(f'{i}.',context ,'->', answer, w)
#     break


frequency_weights:SCALE = {
    'never':-4,
    'very rarely':-3,
    'seldom':-2,
    'rarely':-2,
    'frequently':2,
    'often':2,
    'very frequently':3,
    'always':4,
}

intensifiers_fraction_without_none:SCALE={
            "few":1,
            "some":2,
            "many":3,
            "most":4,
            "all":5,
        }

certainty_weights:SCALE = {
    "isn't":-2,
    "can't be":-2,
    "isn't probably":-1,
    'is probably':1,
    'can be':1,
    'is':2,
}

# Load Models

In [ ]:
p = 'valhalla/distilbart-mnli-12-6'
mnli = pipeline("zero-shot-classification",device=device, model=p)
mnli.model_identifier = p

In [ ]:
 gc.collect()
 torch.cuda.empty_cache()

# Linguastic acceptability

In [ ]:
sentence_embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
cola = pipeline("text-classification","mrm8488/deberta-v3-small-finetuned-cola", device=device)

import os
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu

def linguistic_acceptabilities(q, index, scale, question_name, student_id, output_path=Path(''), save_to_file=False):
    score_by_cola_lst=[]
    score_of_semantic_distance_lst=[]
    score_by_bleu_lst=[]
    kmap_lst=[]
    question_name_lst=[]
    description = q._descriptor
    strFactor=description['Factor']
    strOrdinal=str(description.get('Ordinal', 0))
    ##cleaning the string to get the original question
    strOriginal= description['Original']
    strOriginal = 'none' if strOriginal is None else strOriginal
    strOriginal=strOriginal.replace(strFactor,'',1)
    strOriginal=strOriginal.replace(strOrdinal,'',1)
    strOriginal=strOriginal.replace('.','',1)
    strOriginal=strOriginal.strip() #the original question
    rows = []

    partial_internal_consistency = partial(q.internal_consistency, filter={}, index=index , scale=scale)
    try:
        silhouette_score = partial_internal_consistency(measure='silhouette_score', metric='correlation')
    except Exception as e:
        print(e)
        print('silhouette_score is set to -1')
        silhouette_score = -1

    if hasattr(q, 'linguistic_acceptability'):
        q.linguistic_acceptability['silhouette_score'] = silhouette_score
        return q.linguistic_acceptability

    for kmap in q._keywords_map:
        score = {}
        score['question_name'] = question_name
        context = q._context_template.format_map(kmap)
        answer = q._answer_template.format_map(kmap)
        score['original_question'] = strOriginal


        cola_score = cola(context +" "+ answer)[0].get('score')
        score['cola_score'] = cola_score
        score['param'] = kmap
        strPermutation= context +" "+ answer
        # sentences = [context +" "+ answer]
        score['question_permutation'] = strPermutation
        #Compute embedding for both lists
        embeddings1 = sentence_embedding_model.encode(strOriginal, convert_to_tensor=True)
        embeddings2 = sentence_embedding_model.encode(strPermutation, convert_to_tensor=True)

        #Compute cosine-similarities
        cosine_scores = util.cos_sim(embeddings1, embeddings2)
        score['semantic_similarity'] = cosine_scores.item()

        score['silhouette_score'] = silhouette_score
        rows.append(score)


    filename = output_path / 'linguistic_acceptabilities.csv'
    df = pd.DataFrame(rows)
    df['student_id'] = student_id
    df = df[['student_id', 'question_name','original_question', 'param','question_permutation','cola_score','semantic_similarity','silhouette_score']]
    if save_to_file:
        if filename.exists():
            df.to_csv(filename, index=False, header=None, mode='a', encoding='utf-8-sig')
        else:
            df.to_csv(filename, index=False, encoding='utf-8-sig')
#     print(f"Linguistic acceptabilities saved in {filename}")
    q.linguistic_acceptability = df
    return df

#import questionaire

In [ ]:
from qpsychometric.personality_traits.compassion_scale import compassion_scale_questionnaire

cs_qmnli_df = compassion_scale_questionnaire['QMNLI']
cs_questions = cs_qmnli_df.get_questions()
assert len(cs_questions) >= 24, "Compassion Scale appears to have fewer than 24 items."

# unpack first 24 items (order preserved)
C1, C2, C3, C4, C5, C6, C7, C8, C9, C10, C11, C12, C13, C14, C15, C16, C17, C18, C19, C20, C21, C22, C23, C24 = cs_questions[:24]

Q1s  = split_question(C1,  index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C1().get_filter_for_postive_keywords(['frequency'])})
Q2s  = split_question(C2,  index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C2().get_filter_for_postive_keywords(['frequency'])})
Q3s  = split_question(C3,  index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C3().get_filter_for_postive_keywords(['frequency'])})
Q4s  = split_question(C4,  index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C4().get_filter_for_postive_keywords(['frequency'])})
Q5s  = split_question(C5,  index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C5().get_filter_for_postive_keywords(['frequency'])})
Q6s  = split_question(C6,  index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C6().get_filter_for_postive_keywords(['frequency'])})
Q7s  = split_question(C7,  index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C7().get_filter_for_postive_keywords(['frequency'])})
Q8s  = split_question(C8,  index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C8().get_filter_for_postive_keywords(['frequency'])})
Q9s  = split_question(C9,  index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C9().get_filter_for_postive_keywords(['frequency'])})
Q10s = split_question(C10, index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C10().get_filter_for_postive_keywords(['frequency'])})
Q11s = split_question(C11, index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C11().get_filter_for_postive_keywords(['frequency'])})
Q12s = split_question(C12, index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C12().get_filter_for_postive_keywords(['frequency'])})
Q13s = split_question(C13, index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C13().get_filter_for_postive_keywords(['frequency'])})
Q14s = split_question(C14, index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C14().get_filter_for_postive_keywords(['frequency'])})
Q15s = split_question(C15, index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C15().get_filter_for_postive_keywords(['frequency'])})
Q16s = split_question(C16, index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C16().get_filter_for_postive_keywords(['frequency'])})
Q17s = split_question(C17, index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C17().get_filter_for_postive_keywords(['frequency'])})
Q18s = split_question(C18, index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C18().get_filter_for_postive_keywords(['frequency'])})
Q19s = split_question(C19, index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C19().get_filter_for_postive_keywords(['frequency'])})
Q20s = split_question(C20, index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C20().get_filter_for_postive_keywords(['frequency'])})
Q21s = split_question(C21, index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C21().get_filter_for_postive_keywords(['frequency'])})
Q22s = split_question(C22, index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C22().get_filter_for_postive_keywords(['frequency'])})
Q23s = split_question(C23, index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C23().get_filter_for_postive_keywords(['frequency'])})
Q24s = split_question(C24, index=["index"], scales=["frequency"], softmax=softmax_files,
                      filters={'unfiltered':{}, "positiveonly": C24().get_filter_for_postive_keywords(['frequency'])})




# Run Questionnaires on models

## Utility functions

In [ ]:
def question_attributes(q):
    score = {}
    score['questionnair']=q._descriptor['Questionnair']
    score['factor']=q._descriptor['Factor']
    score['ordinal']=q._descriptor['Ordinal']
    score['scale']=q._descriptor['scale']
    score['index']=q._descriptor['index']
    score['filter']=q._descriptor['filter']
    score['softmax'] = q._descriptor['softmax']
    score["original"] = q._descriptor['Original']
    score['Q'] = f"{score['questionnair']}{score['factor']}{score['ordinal']}"
    score['context_template'] = q._context_template
    score['answer_template'] = q._answer_template
    score['dimensions'] = q._dimensions
    score['model'] = q.model.model_identifier if q.model else ""
    return score

def get_question_features(q, student_id='student_id', output_path=Path(''), save_to_file=False):
    score = question_attributes(q)
    score['mean_score'] = q.mean_score()
    index= q._index
    scale= q._scale
    linguistic_df = linguistic_acceptabilities(q, index=index, scale=scale,question_name=score['Q'], student_id=student_id,
                                               output_path=output_path, save_to_file=save_to_file)
    row = linguistic_df[['cola_score','silhouette_score']].mean(axis=0)
    row_dict = dict(row)
    row_dict['semantic_similarity'] = linguistic_df['semantic_similarity'].quantile(0.75)
    score = score | row_dict
    return score

def extract_epoch(model_path):
    if 'epoch-' in model_path.name:
        i = model_path.name.find('epoch-')
        j = model_path.name.find('_', i)
        if j > 0:
            epoch = int(model_path.name[i+len('epoch-'):j])
        else:
            epoch = int(model_path.name[i+len('epoch-'):])

    elif 'checkpoint-' in model_path.name:
        i = model_path.name.find('checkpoint-')
        j = model_path.name.find('_', i)
        if j > 0:
            epoch = int(model_path.name[i+len('checkpoint-'):j])
        else:
            epoch = int(model_path.name[i+len('checkpoint-'):])
    else:
        epoch = 0
    return epoch

def extract_run(model_path):
    try:
        if 'run' in model_path.name:
            for part in model_path.name.split('_'):
                if 'run' in part:
                    return int(part.replace('run', ''))
        else:
            return -1
    except Exception as e:
        print(e)
        return -1

import json

def get_mnli_score(checkpoint_path):
    mnli_score_path = checkpoint_path / 'all_results.json'
    if not mnli_score_path.exists():
        mnli_score_path = checkpoint_path.parent / (checkpoint_path.name + '_mnli_eval') / 'all_results.json'
    if mnli_score_path.exists():
        with open(mnli_score_path) as f:
            return json.load(f)["eval_accuracy"]
    else:
        return -1


def run_questions(questions, mnli_checkpoint, train_process, fintune_dataset, q_range=[5, 0]):
    rows = []
    checkpoint = Path(mnli_checkpoint.model_identifier)
    for q_raw in tqdm(questions):
        T = time.time()
        q = q_raw.run(mnli_checkpoint)
        T = time.time()
        score = get_question_features(q)
        score['epoch'] = extract_epoch(checkpoint)
        score['train_process'] = train_process
        score['dataset'] = fintune_dataset
        score['run'] = extract_run(checkpoint.parent)
        score['mnli_score'] = get_mnli_score(checkpoint)
        score['range'] = (q._weights_flat.min(), q._weights_flat.max())
        score['ASI_score'] = np.interp(score['mean_score'], [q._weights_flat.min(), q._weights_flat.max()], q_range)
        rows.append(score)
        gc.collect()
        torch.cuda.empty_cache()
    return rows


def calc_scores(questions, checkpoint, output_path, train_process, fintune_dataset, q_range=[5, 0]):
    fix_config(checkpoint)
    mnli_checkpoint = pipeline("zero-shot-classification", str(checkpoint), device=device)
    mnli_checkpoint.model_identifier = str(checkpoint)
    rows = run_questions(questions, mnli_checkpoint, train_process, fintune_dataset=fintune_dataset, q_range=q_range)
    return rows

def add_epochs_to_rows(rows, mlm_epoch, mnli_checkpoint):
    for score in rows:
        score['mlm_epoch'] = mlm_epoch
        score['mnli_checkpoint'] = mnli_checkpoint
    return rows


def write_to_csv(rows, output_path):
    old_score_hostile_df = pd.DataFrame(rows)
    if output_path.exists():
        old_score_hostile_df.to_csv(output_path, index=False, header=None, mode='a')
    else:
        old_score_hostile_df.to_csv(output_path, index=False)

def fix_config(checkpoint):
    if checkpoint.exists():
        with open(checkpoint / 'config.json') as f:
            d1 = json.load(f)
        d1['id2label'] = {'0': 'entailment', '1': 'neutral', '2': 'contradiction'}
        d1['label2id'] = {'contradiction': 2, 'entailment': 0, 'neutral': 1}
        with open(checkpoint / 'config.json', 'w') as f:
            json.dump(d1, f)
    else:
        print(checkpoint, '#### Not exists ####')

def calc_for_all_models(Qs, q_range= [5, 0]):
    all_rows = []
    for p in tqdm(mnli_pipelines):
        print(p)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            rows = calc_scores(Qs, Path(p),  Path(p), '->'.join(['base']), 'hostile',
                               use_base_model=False, q_range=q_range)
            rows = add_epochs_to_rows(rows, 0, 0)
            all_rows += rows
    return pd.DataFrame(all_rows)

## Run Questions

In [ ]:
result_path = Path('results/')
if not result_path.exists():
    os.makedirs(result_path)

In [ ]:
mnli_pipelines = [
    'typeform/distilbert-base-uncased-mnli',
    'typeform/mobilebert-uncased-mnli',
    'cross-encoder/nli-roberta-base',
    'cross-encoder/nli-deberta-base',
    'cross-encoder/nli-distilroberta-base',
    'cross-encoder/nli-MiniLM2-L6-H768',
    'navteca/bart-large-mnli',
    'digitalepidemiologylab/covid-twitter-bert-v2-mnli',
    'joeddav/bart-large-mnli-yahoo-answers',
    'Narsil/deberta-large-mnli-zero-cls',
    'microsoft/deberta-large-mnli',
    'microsoft/deberta-base-mnli',
    'Alireza1044/albert-base-v2-mnli',
    'yoshitomo-matsubara/bert-large-uncased-mnli',
    'yoshitomo-matsubara/bert-base-uncased-mnli',
    'yoshitomo-matsubara/bert-base-uncased-mnli_from_bert-large-uncased-mnli',
    'valhalla/distilbart-mnli-12-6',
]


In [ ]:
from collections import defaultdict

questions = (
    Q1s + Q2s + Q3s + Q4s + Q5s + Q6s +
    Q7s + Q8s + Q9s + Q10s + Q11s + Q12s +
    Q13s + Q14s + Q15s + Q16s + Q17s + Q18s +
    Q19s + Q20s + Q21s + Q22s + Q23s + Q24s
)


update = True

output_path = result_path / f'cs_mnli_check1.csv'
pipelines = mnli_pipelines

if output_path.exists():
    temp_df = pd.read_csv(output_path)
    indexes = temp_df.groupby(['model', 'Q']).count().index.values
    used_models = defaultdict(set)
    for k, v in indexes:
        used_models[k].add(v)
else:
    used_models = {}


for p in tqdm(pipelines):
    print(p)
    if get_mnli_score(Path(p)) < 0.7 and p not in mnli_pipelines:
        print('Skip:', p)
        continue
    with warnings.catch_warnings():
        try:
            warnings.simplefilter("ignore")
            if p in used_models and not update:
                pipline_questions = []
                for q in questions:
                    if question_attributes(q)['Q'] not in used_models[p]:
                        pipline_questions.append(q)
                    else:
                        print('skip', p, question_attributes(q)['Q'])
            else:
                pipline_questions = questions

            rows = calc_scores(pipline_questions, Path(p),  output_path, '->'.join(['base']), 'hostile',)
            rows = add_epochs_to_rows(rows, 0, 0)
            write_to_csv(rows, output_path)
            gc.collect()
            torch.cuda.empty_cache()
        except Exception as e:
            print(e)


df = pd.read_csv(output_path)
df = df.drop_duplicates(subset=['filter','softmax','model','Q'], keep='last')
df.to_csv(output_path, index=False)

# Validations

In [ ]:
# ---------------- helper: load_results (robust) ----------------
def load_results(csv_path, softmax=None, positiveonly=False, value='mean_score', index='model'):
    """
    Reads a results CSV and returns a WIDE dataframe:
      rows = index (e.g., model), columns = Q, values = `value` (mean across duplicates).
    Optional filters:
      - softmax: list of slot names to keep (matches df['softmax'])
      - positiveonly: keep only rows where df['filter'] == 'positiveonly'
    Also ignores rows with silhouette_score == -1 (if column exists).
    """
    df = pd.read_csv(csv_path)

    # Ensure the row id exists
    if index not in df.columns:
        # Fallback: try common alternatives
        for alt in ['model_id', 'mnli_checkpoint', 'checkpoint', 'student_id']:
            if alt in df.columns:
                index = alt
                break
        else:
            raise ValueError(f"Index column '{index}' not found and no fallback present.")

    # --- softmax filtering ---
    if softmax:
        if 'softmax' not in df.columns:
            raise ValueError("Requested softmax filtering but 'softmax' column not found in CSV.")
        # Accept either single string or list; keep rows where softmax equals any of them
        softmax = [str(s) for s in (softmax if isinstance(softmax, (list, tuple)) else [softmax])]
        df = df[df['softmax'].astype(str).isin(softmax)]

    # --- silhouette filter (ignore invalid = -1) ---
    if 'silhouette_score' in df.columns:
        df = df[df['silhouette_score'] > -1]

    # --- positive-only filter ---
    if positiveonly and 'filter' in df.columns:
        df = df[df['filter'] == 'positiveonly']
    elif 'filter' in df.columns:
        # drop rows explicitly marked as 'unfiltered' only if you need to
        pass

    # sanity: required columns
    required_cols = {index, 'Q', value}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}")

    # Pivot to wide; average duplicates if any
    df = df.groupby([index, 'Q'], as_index=False)[value].mean()
    results_df = df.pivot(index=index, columns='Q', values=value)

    return results_df


In [ ]:
# ---------------- Compassion Scale configuration (LIVE) ----------------

softmax_compassion = []          # e.g., [] or ['frequency'] if you actually use a softmax slot
positiveonly = False             # keep symmetric evidence unless you want only positive cues

# Common 6-factor layout (rename if your instrument differs)
all_factors = [
    'Kindness',
    'CommonHumanity',
    'Mindfulness',
    'Indifference_R',
    'Separation_R',
    'Disengagement_R'
]

# Path to your Compassion results CSV (parallel to your Big Five path)
q_path = result_path / 'cs_mnli_check1.csv'

# If your pipeline iterates over filters, keep this structure
all_filters = [softmax_compassion]


## Semantic Validation

In [ ]:
# Metrics to aggregate per item (Q)
cols = ['semantic_similarity', 'cola_score', 'silhouette_score']

# Choose filters: either a single list like softmax_big5, or iterate all_filters
filters_to_run = all_filters if 'all_filters' in globals() else [softmax_big5]  # fallback

results = []
for softmax_filter in filters_to_run:
    # load_results returns wide: rows = index (e.g., model), cols = Q, values = metric
    q_means = [load_results(q_path, softmax=softmax_filter, positiveonly=False, value=v).mean(axis=0)
               for v in cols]
    df_block = pd.concat(q_means, axis=1)   # index = Q, columns = metrics (in original order)
    df_block.columns = cols
    # Keep track of which filter produced it (optional)
    df_block['softmax'] = ','.join(map(str, softmax_filter)) if softmax_filter else ''
    results.append(df_block)

# Stack all filters (if only one filter, this is just the same frame)
linguistic_acceptability_df = pd.concat(results, axis=0)

# Save with Q as a proper column
out_path = result_path / 'linguistic_acceptability.csv'
linguistic_acceptability_df.reset_index(names='Q').to_csv(out_path, index=False)

linguistic_acceptability_df


In [ ]:
df_clean = linguistic_acceptability_df.replace('', np.nan)
print(df_clean.mean(numeric_only=True))


In [ ]:
df_clean.std(numeric_only=True)

## Internal Consistency

In [ ]:
# ---------- Compassion factor keywords ----------
COMPASSION_FACTOR_KEYWORDS = {
    'Kindness':         ['kind', 'kindness', 'comp_k', 'cs_k', 'c_k'],
    'CommonHumanity':   ['commonhumanity', 'common_humanity', 'hum', 'humanity', 'comp_ch', 'cs_ch', 'c_ch'],
    'Mindfulness':      ['mind', 'mindful', 'mindfulness', 'comp_m', 'cs_m', 'c_m'],
    'Indifference_R':   ['indiff', 'indifference', 'indifference_r', 'comp_ind', 'cs_ind', 'c_ind'],
    'Separation_R':     ['sep', 'separation', 'separation_r', 'comp_sep', 'cs_sep', 'c_sep'],
    'Disengagement_R':  ['diseng', 'disengagement', 'disengagement_r', 'comp_dis', 'cs_dis', 'c_dis'],
}

def get_factor_sub_features_compassion(factors, data_df, factor_map: dict | None = None):
    """
    Return columns in data_df that belong to any Compassion factor in `factors`.
    - If factor_map is provided: use it (e.g., {'CS_K_01':'Kindness', ...}).
    - Else: match by keywords (case-insensitive) and token presence.
    """
    cols = list(data_df.columns)
    if factor_map:
        wanted = set(factors)
        out = [c for c in cols if factor_map.get(str(c)) in wanted]
    else:
        selected = []
        for f in factors:
            keys = COMPASSION_FACTOR_KEYWORDS.get(f, [])
            for c in cols:
                name = f"_{str(c).lower()}_"
                if any(k in name for k in keys) or f"_{f.lower()}_" in name:
                    selected.append(c)
        # de-dupe while preserving original col order
        seen, out = set(), []
        for c in cols:
            if c in selected and c not in seen:
                out.append(c); seen.add(c)
    return out

# ---------- Compassion config ----------
all_factors          = ['Kindness', 'CommonHumanity', 'Mindfulness', 'Indifference_R', 'Separation_R', 'Disengagement_R']
softmax_compassion   = []          # keep [] unless you actually have a softmax slot (e.g., ['frequency'])
positiveonly         = False       # mirror Big5 unless you want only positive cues
q_path               = result_path / 'cs_mnli_check1.csv'  # parallel to your Big5 file

# --- Compassion factor means + Spearman correlations ---
value = 'mean_score'

# Load wide results (rows = id, cols = Q). If you have multiple filters, add them to this list.
results = []
for softmax_filter in [softmax_compassion]:
    results.append(
        load_results(
            q_path,
            softmax=softmax_filter,
            positiveonly=positiveonly,
            value=value
        )
    )

# Concatenate wide frames column-wise (same index expected)
data_df = pd.concat(results, axis=1)

# Build factor means
filtered_df = pd.DataFrame(index=data_df.index)  # keep row alignment

for factor in all_factors:
    feature_subset = get_factor_sub_features_compassion([factor], data_df)  # or pass factor_map=... if you have one
    if len(feature_subset) > 0:
        filtered_df[factor] = data_df[feature_subset].mean(axis=1, skipna=True)

# Drop empty columns, if any
filtered_df = filtered_df.dropna(axis=1, how='all')

# If nothing was matched, stop early with a helpful note
if filtered_df.shape[1] == 0:
    print("No Compassion factor columns were matched. Check your column names, keyword lists, or provide a factor_map.")
else:
    # Spearman correlations (with p-values via Pingouin)
    if filtered_df.shape[1] >= 2 and filtered_df.shape[0] >= 3:
        corr_with_p = pg.rcorr(filtered_df, method='spearman')  # returns r and p-values
        print(corr_with_p)
    else:
        # fallback: plain matrix if too few rows for rcorr, or just informative print
        if filtered_df.shape[1] >= 2:
            print("Not enough rows for pg.rcorr (need ≥3). Showing plain Spearman matrix:")
            print(filtered_df.corr(method='spearman'))
        else:
            print("Need at least 2 factor columns to compute correlations.")


In [ ]:
# --- config for Compassion ---
all_factors_comp = ['Kindness', 'CommonHumanity', 'Mindfulness', 'Indifference_R', 'Separation_R', 'Disengagement_R']
softmax_compassion = []                # keep [] unless your items have a softmax slot (e.g., ['frequency'])
positiveonly_compassion = False        # mirror Big Five unless you want only positive cues
value = 'mean_score'
q_path = result_path / 'cs_mnli_check1.csv'

# --- helper: select columns by factor (edit keyword lists if your prefixes differ) ---
COMPASSION_FACTOR_KEYWORDS = {
    'Kindness':         ['kind', 'kindness', 'comp_k', 'cs_k', 'c_k'],
    'CommonHumanity':   ['commonhumanity', 'common_humanity', 'hum', 'humanity', 'comp_ch', 'cs_ch', 'c_ch'],
    'Mindfulness':      ['mind', 'mindful', 'mindfulness', 'comp_m', 'cs_m', 'c_m'],
    'Indifference_R':   ['indiff', 'indifference', 'indifference_r', 'comp_ind', 'cs_ind', 'c_ind'],
    'Separation_R':     ['sep', 'separation', 'separation_r', 'comp_sep', 'cs_sep', 'c_sep'],
    'Disengagement_R':  ['diseng', 'disengagement', 'disengagement_r', 'comp_dis', 'cs_dis', 'c_dis'],
}

def get_factor_sub_features_compassion(factors, data_df, factor_map: dict | None = None):
    cols = list(data_df.columns)
    if factor_map:
        wanted = set(factors)
        return [c for c in cols if factor_map.get(str(c)) in wanted]
    selected = []
    for f in factors:
        keys = COMPASSION_FACTOR_KEYWORDS.get(f, [])
        for c in cols:
            name = f"_{str(c).lower()}_"
            if any(k in name for k in keys) or f"_{f.lower()}_" in name:
                selected.append(c)
    seen, out = set(), []
    for c in cols:
        if c in selected and c not in seen:
            out.append(c); seen.add(c)
    return out

# --- load results (wide: rows=id, cols=Q) ---
results = []
for softmax_filter in [softmax_compassion]:
    results.append(
        load_results(q_path,
                     softmax=softmax_filter,
                     positiveonly=positiveonly_compassion,
                     value=value)
    )
data_df = pd.concat(results, axis=1)

# --- Cronbach's alpha per factor ---
import pingouin as pg

print('Cronbach Alpha (Compassion):')
for f in all_factors_comp:
    cols = get_factor_sub_features_compassion([f], data_df)   # pass factor_map=... if you have an explicit map
    if len(cols) < 2 or data_df[cols].shape[0] < 2:
        print(f'{f}, Alpha: skipped (need ≥2 items & ≥2 rows). Found {len(cols)} items.')
        continue
    alpha, _ = pg.cronbach_alpha(data=data_df[cols])
    print(f'{f}, Alpha: {float(alpha):.3f}')

# --- Overall Compassion alpha (all items combined) ---
all_cols = get_factor_sub_features_compassion(all_factors_comp, data_df)
if len(all_cols) >= 2 and data_df[all_cols].shape[0] >= 2:
    alpha_all, _ = pg.cronbach_alpha(data=data_df[all_cols])
    print(f'Compassion Overall, Alpha: {float(alpha_all):.3f}')
else:
    print('Compassion Overall, Alpha: skipped (need ≥2 items & ≥2 rows).')
